# Connecting to the ENTSO-E API

**GridForecast team guide** · Author: Jalil Muñoz Robles

This notebook explains, step by step, how we connect to the **ENTSO-E Transparency Platform** to download European electricity grid data: which libraries we use, how to authenticate securely, and how to fetch specific data (demand, generation, renewable forecasts, and prices).

> This is learning and reference material. The production script that automates this flow lives in `src/ingest.py`.

## 1. The libraries we use

Three pieces, each with a clear job:

- **`entsoe-py`** — a Python client that wraps the ENTSO-E REST API. You ask for data and it returns it directly as pandas objects, so you don't have to parse the API's raw XML. We use its `EntsoePandasClient` class.
- **`pandas`** — the standard library for tabular data (DataFrames) and time series. ENTSO-E returns series indexed by date/time, and pandas is the natural tool to work with them.
- **`python-dotenv`** — loads environment variables from a local `.env` file. We use it to read the token without ever writing it in the code.

In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from entsoe import EntsoePandasClient

print("Libraries imported successfully")
print("pandas version:", pd.__version__)

## 2. Load the token securely

API access requires a **token** (a kind of password). The golden rule: **the token is never written in the code or pushed to GitHub.** It lives in a `.env` file at the project root, protected by `.gitignore`.

`find_dotenv()` looks for the `.env` file by walking up from the current folder, so it works even if you run the notebook from `notebooks/`.

In [ ]:
load_dotenv(find_dotenv())
token = os.getenv("ENTSOE_API_TOKEN")

if not token or token == "your_token_here":
    raise SystemExit(
        "Token not found. Copy .env.example to .env and paste your ENTSO-E token."
    )

# We never print the token, not even partially.
print("Token loaded successfully")

## 3. Create the client

With the token, we create the `EntsoePandasClient` object. This client is our "door" to all the data: every query will be a method on this object.

In [ ]:
client = EntsoePandasClient(api_key=token)
print("ENTSO-E client ready")

## 4. Timezone-aware dates (a detail that matters)

ENTSO-E **requires** timezone-aware timestamps. "January 1st" isn't enough; you must say "January 1st in Europe/Madrid time". This avoids ambiguity around daylight-saving changes and between time zones.

We define a window of the last 7 days. `pd.Timestamp.now(tz=...)` gives the current moment with a timezone, and we subtract a window with `pd.Timedelta`.

In [ ]:
TIMEZONE = "Europe/Madrid"

end = pd.Timestamp.now(tz=TIMEZONE)
start = end - pd.Timedelta(days=7)

print("From:", start)
print("To:  ", end)

## 5. Fetch specific data

Here's the heart of it: each data type is a different method on the client. All of them take a **country code** and a **date range**. Let's look at the four most useful for GridForecast.

### 5.1 Electricity demand (actual load)

This is our main target variable: how much electricity the system consumed. `query_load` returns the measured actual demand.

In [ ]:
load = client.query_load("ES", start=start, end=end)
load.head()

### 5.2 Generation by technology type

How much electricity was generated and from what source (gas, nuclear, hydro, wind, solar...). With `psr_type=None` it returns all technologies.

In [ ]:
generation = client.query_generation("ES", start=start, end=end, psr_type=None)
generation.head()

### 5.3 Wind and solar forecast

The official renewable generation forecast. It's key for the project: part of what we want to predict and compare against.

In [ ]:
renewables = client.query_wind_and_solar_forecast("ES", start=start, end=end, psr_type=None)
renewables.head()

### 5.4 Day-ahead market prices

The hourly wholesale electricity price. It enriches the analysis and can serve as an explanatory variable.

In [ ]:
prices = client.query_day_ahead_prices("ES", start=start, end=end)
prices.head()

## 6. Switching countries

ENTSO-E covers dozens of countries; you just change the code. Some examples:

- `"ES"` — Spain
- `"PT"` — Portugal
- `"FR"` — France
- `"DE_LU"` — Germany-Luxembourg
- `"IT"` — Italy

This lets the project scale (for example, comparing grid behavior across countries) without changing data source.

In [ ]:
# Example: Portugal's demand over the same range
load_pt = client.query_load("PT", start=start, end=end)
load_pt.head()

## 7. Explore and save

With the data in a pandas DataFrame, exploring and saving it is straightforward.

In [ ]:
print("Shape (rows, columns):", load.shape)
print("Date range:", load.index.min(), "->", load.index.max())
load.describe()

In [ ]:
# Save to Parquet (an efficient, versionable format).
# The data/ folder is in .gitignore: data is NOT pushed to the repo.
from pathlib import Path

Path("../data").mkdir(exist_ok=True)
load.to_parquet("../data/es_load_example.parquet")
print("Saved to data/es_load_example.parquet")

## 8. Best practices and notes

- **Never push `.env`.** The token is personal; each member generates their own.
- **Fair use:** the API is a shared resource. Avoid downloading huge ranges in a loop; request only what you need.
- **Attribution:** the data is from ENTSO-E (EU Regulation 543/2013). ENTSO-E must be cited as the source in the project.
- **Timezone:** always use timezone-aware timestamps; it's the most common cause of query errors.

---

With this you have the full map of how GridForecast gets its data. The script `src/ingest.py` automates this flow; the other methods you saw here are the next bricks of the pipeline.